Import the library and create a client for interacting with the API.

In [50]:
import usnan
client = usnan.USNANClient()

Now let's fetch information on the facilities in the NAN network. Note that if you run this twice

In [71]:
facilities = client.facilities.list()
print(f'Facilities: {facilities}')
facility = facilities[0]
print(facility)

Facilities: [Facility('UGA-CCRC'), Facility('UCHC-Mullen'), Facility('Wisc-NMRFAM'), Facility('OSU-CCIC'), Facility('UNL-MAC'), Facility('UCSC-NMR-Facility')]
Facility: Complex Carbohydrate Research Center (CCRC — UGA)
Short Name: CCRC - UGA
Identifier: UGA-CCRC
Institution: The University of Georgia (UGA)
URL: http://nmr.ccrc.uga.edu
Description: The CCRC-UGA NMR facility is located in the Complex Carbohydrate Research Center which hosts scientists studying molecular structures, interactions, and metabolism of mammalian, plant and microbial systems, with an emphasis on glycobiology and metabolomics. As part of NAN, we are fully networked for remote access and can provide training and assistance for many different applications of NMR. CCRC personnel can assist with purification, extraction, and sample preparation protocols to ensure that your samples become NMR-ready.

We are equipped with six Bruker spectrometers (600-1100 MHz). Specialized probes include a highly sensitive 1.7 mm TCI

What exactly does a facility look like?

In [52]:
print([_ for _ in dir(facility) if not _.startswith('_')])
print(facility.addresses)

['addresses', 'color', 'contacts', 'description', 'from_dict', 'from_identifier', 'identifier', 'institution', 'logo', 'long_name', 'services', 'short_name', 'spectrometers', 'staff', 'url', 'webpages']
[Address(address_type=['Physical', 'Mailing', 'Shipping'], address1='315 Riverbend Road', address2=None, address3=None, city='Athens', state='Georgia', zipcode='30602', zipcode_ext=None, country='United States of America')]


The facility logo?

In [53]:
import base64
from IPython.display import HTML
svg_data = base64.b64decode(facility.logo).decode("utf-8")
HTML(f'''
<div style="width:250px; height:auto;">
    {svg_data}
</div>
''')

What about the spectrometers?

In [54]:
print(facility.spectrometers)
spectrometer = facility.spectrometers[0]
print(spectrometer)

[Spectrometer('1f8a0d2a-94bd-4604-825c-91e2d854b336), Spectrometer('6050972c-5bd6-4f7b-bcc1-b784f33a7aab), Spectrometer('813edc25-e42f-4fb9-a7f7-69fa5e2f6a2c), Spectrometer('6fbe10d5-86fa-4cd3-a9c1-5650bafac2a2), Spectrometer('b684f5ff-8409-4d7e-8316-43da7e6b25c2), Spectrometer('ac266789-3e47-483a-b565-2fdf6d8aa3f1), Spectrometer('056ba09b-45ef-43a5-8b11-d21b2dc5bff2)]
Spectrometer: br1100
Identifier: 1f8a0d2a-94bd-4604-825c-91e2d854b336
Status: Operational
Commissioned: 2024
Field Strength: 1100 MHz
Bore: 51 mm
Pumped: Yes
Console: Bruker AVANCE NEO
Configured: 2024
Channels: 5
Receivers: 4
OS: Alma 9.2
Software: Topspin
  Version: 4.x.x
Sample Changer: Bruker SampleJet 30/5 Cooled (30 spinners)
Installed Probe: 5mm cryoprobe TXO C13|N15(H1|H2) Z-axis
Compatible Probes:
  3mm cryoprobe TCI H1(C13|N15|H2) Z-axis
  5mm cryoprobe TXO C13|N15(H1|H2) Z-axis
  5mm TXI H1(C13|N15|H2) Z-axis
Public Access: Yes


The spectrometers were not loaded from the API until the moment I accessed them. If I re-run the above, it is significantly faster.

And the probes in a spectrometer?

In [55]:
print(f'Compatible probes: {spectrometer.compatible_probes}')
print(f'Installed probe: {spectrometer.installed_probe}')

Compatible probes: [Probe('300795f0-34b3-4888-b3fb-2aef1faa4290'), Probe('4f31b022-42d1-47a6-999e-8670145c9ca2'), Probe('45a56dd5-2d0c-4d41-ba36-67a074088e74')]
Installed probe: Probe: 5mm cryoprobe TXO C13|N15(H1|H2) Z-axis
Identifier: 4f31b022-42d1-47a6-999e-8670145c9ca2
Model: Bruker TXO
Serial: Z164847_0001
Status: Operational
Type: Solution
Sample Diameter: 5 mm
Cooling: Helium
1H Field Strength: 1100 MHz
Temperature Range: -20°C to 80°C
Gradient: Yes (Z: 50)
Facility: CCRC - UGA
Installed on: br1100 since 2024-08-28T00:00:00-04:00
Channels (4):
  Ch 1 (Inner: True)
    C13
  Ch 2 (Inner: True)
    N15
  Ch 3 (Outer: True)
    H1
  Ch 4 (Outer: True)
    H2


The channels in a probe?

In [56]:
print(repr(spectrometer.installed_probe.channels[0]))

Channel(ch_number=1, amplifier_cooled=True, inner_coil=True, outer_coil=False, min_frequency_nucleus=None, max_frequency_nucleus=None, broadband=False, nuclei=[Nucleus(nucleus='C13', sensitivity_measurements=[])])


Great, I can get facility, spectrometer, and probe metadata. But what about experiments!?!?

In [76]:
search_config = (usnan.models.SearchConfig()
                 .add_filter('is_knowledgebase', value=True, match_mode='equals')
                 .add_filter('num_dimension', value=2, match_mode='equals'))
results = client.datasets.search(search_config)
print(results)

<generator object DatasetsEndpoint.search at 0x7a961a938c10>


A... generator object?

In [77]:
print(next(results))

JRES for human plasma collected on 600 MHz (5-mm) 'RedCross_plasma_human_5004450847' (2021-12-10T18:11:14-05:00 - 2021-12-10T13:18:14-05:00)


In [78]:
datasets = list(results)
print(len(datasets))

124


A search result is an object that will produce datasets as it is iterated over. If there are 2 million results, it will fetch them as you process them rather than all at once.

In [82]:
dataset = datasets[1]
print(dataset)
print(repr(dataset.spectrometer.facility))

HSQC_15N_900.data/2 (2021-12-16T14:08:03-05:00 - 2021-12-16T14:19:36-05:00)
Facility('UGA-CCRC')


Now let's take a look at the properties of a dataset...

In [61]:
print(dataset)

HSQC for human plasma collected on 600 MHz (5-mm) 'RedCross_plasma_human_5004450847' (2021-12-13T21:09:42-05:00 - 2021-12-13T16:16:42-05:00)


Can we download a dataset?

In [62]:
client.datasets.download([dataset.id], '/tmp/dataset')

ValueError: Target directory '/tmp/dataset' is not empty

Now we've downloaded the dataset, let's take a look.

In [63]:
import os
print(os.listdir('/tmp/dataset'))
print(os.listdir('/tmp/dataset/20211210131814_br600_jresgpprqf'))

['experiments.csv', '20211210131814_br600_jresgpprqf']
['provenance.provn', 'identity.xml', 'JRES_br600_serum_human.data', 'sample_metadata.xml', 'supplemental_data.csv', 'experiment.csv']
